# YOLO12s + L-Ghost + EMA-32 — Aceh Road Damage Detection 2026

Model pretrained empat kelas: `D00`, `D10`, `D20`, dan `D40`.
Alur mengikuti notebook baseline: audit dataset → train → validation → test → ZIP.

Dataset dan hyperparameter mengikuti `aceh-ardd-yolo12-baseline (1).ipynb`: 160 epoch, imgsz 640,
batch 16, nbs 64, SGD, lr0 0.01, momentum 0.937, weight_decay 0.0005, patience 0, seed 42, deterministic=True.
Source model dari branch `feat/yolo12s-lghost-ema32`, commit `14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e`.

LG menggunakan G=4 dan RA=0.6, dengan pruning setelah 32, 64, dan 96 epoch.
`LGhostDetectionTrainer` menjalankan training deteksi dengan jadwal pruning dan sinkronisasi mask ModelEMA.
EMA-32 attention digunakan pada P3/P4.
ModelEMA rata-rata bobot Ultralytics tetap terpisah dari EMA attention.

Aktifkan Internet dan satu GPU Kaggle; mulai dari session bersih. Notebook belum dilatih.
Model adalah adaptasi L-GhostNet. Hasil penelitian ditentukan setelah training dan evaluasi dataset Aceh.

In [ ]:
# 1. Environment. Aktifkan GPU dan Internet pada Kaggle.
import json, platform, subprocess, sys, warnings, zipfile
from collections import Counter
from pathlib import Path

if 'ultralytics' in sys.modules:
    raise RuntimeError('Restart session sebelum install agar import memakai repository penelitian.')

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
SOURCE_BRANCH = 'feat/yolo12s-lghost-ema32'
SOURCE_COMMIT = '14cbeb1b9a136c17da4ff0acfe01a3f625cd5e3e'
REPO_DIR = WORKDIR / 'yolo12s-lghost-ema32-ardd26-source'

def run(command, cwd=None):
    print('>>', subprocess.list2cmdline([str(item) for item in command]))
    subprocess.run(command, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', SOURCE_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    origin = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=REPO_DIR, text=True).strip()
    status = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True).strip()
    assert origin == REPO_URL, 'Folder berisi repository berbeda.'
    assert not status, 'Repository memiliki perubahan lokal; gunakan folder clone baru.'
    run(['git', 'fetch', 'origin', SOURCE_BRANCH], cwd=REPO_DIR)
run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert REPO_COMMIT == SOURCE_COMMIT
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))

# Ray pada environment Kaggle kadang meminta trial ID meskipun training ini
# bukan Ray Tune. Abaikan hanya warning spesifik yang tidak memengaruhi YOLO.
warnings.filterwarnings(
    'ignore',
    message=r'`get_trial_id` is meant to only be called.*',
    category=UserWarning,
    module=r'ray\.train\._internal\.session',
)

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.models.yolo.detect.lghost import LGhostDetectionTrainer
from ultralytics.nn.modules import EMA, LGhostBottleneck
from ultralytics.nn.modules.lghost import LearnedGroupConv
from ultralytics.nn.tasks import DetectionModel

WORKDIR = Path('/kaggle/working')
assert Path(ultralytics.__file__).resolve().is_relative_to(REPO_DIR.resolve()), 'Import bukan dari repo clone.'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def log_section(title):
    print(f'\n{"=" * 88}\n{title}\n{"=" * 88}')

log_section('ENVIRONMENT')
print(f'Python      : {platform.python_version()}')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Device      : {DEVICE}')
print(f'GPU         : {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'WARNING: GPU tidak aktif')
print(f'Source branch: {SOURCE_BRANCH}')
print(f'Commit      : {REPO_COMMIT}')

In [ ]:
# 2. Konfigurasi dan audit dataset sebelum training.
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ardd26/datasets_here')
DATA_YAML = WORKDIR / 'ardd26_4class.yaml'
CLASS_NAMES = ['D00', 'D10', 'D20', 'D40']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = 'SGD', 0.01, 0.937, 0.0005
PATIENCE, WORKERS, SEED = 0, 2, 42
EXPERIMENT_NAME = 'yolo12s_lghost_ema32_pretrained_ardd26'
RUNS_DIR = WORKDIR / 'runs'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12s-lghost-ema32.yaml'
assert MODEL_YAML.is_file(), f'Model YAML tidak ditemukan: {MODEL_YAML}'

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset tidak ditemukan: {DATA_ROOT}')

def file_stems(folder, extensions):
    if not folder.is_dir():
        raise FileNotFoundError(f'Folder tidak ditemukan: {folder}')
    return {p.stem.casefold() for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions}

dataset_report = {}
for split in ('train', 'val', 'test'):
    image_dir, label_dir = (DATA_ROOT / split / name for name in ('images', 'labels'))
    image_stems = file_stems(image_dir, IMAGE_EXTENSIONS)
    label_stems = file_stems(label_dir, {'.txt'})
    if image_stems != label_stems:
        raise ValueError(f'Pairing {split} gagal: images={len(image_stems)}, labels={len(label_stems)}')
    counts, empty_labels = Counter(), 0
    for label_path in sorted(label_dir.glob('*.txt')):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        empty_labels += int(not lines)
        for line_number, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                raise ValueError(f'{label_path}:{line_number}: format bukan 5 kolom')
            class_id, coords = int(parts[0]), [float(v) for v in parts[1:]]
            if class_id not in range(4):
                raise ValueError(f'{label_path}:{line_number}: class ID {class_id} di luar 0..3')
            if not all(0 <= v <= 1 for v in coords) or coords[2] <= 0 or coords[3] <= 0:
                raise ValueError(f'{label_path}:{line_number}: bounding box invalid')
            counts[class_id] += 1
    dataset_report[split] = {'pairs': len(image_stems), 'objects': sum(counts.values()), 'empty_labels': empty_labels, 'classes': {CLASS_NAMES[i]: counts[i] for i in range(4)}}

DATA_YAML.write_text(f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
''', encoding='utf-8')

log_section('DATASET AUDIT PASSED')
print(json.dumps(dataset_report, indent=2))
print(f'Total pairs: {sum(v["pairs"] for v in dataset_report.values())}')
print(f'Total objects: {sum(v["objects"] for v in dataset_report.values())}')
print(DATA_YAML.read_text())

In [ ]:
# 3. Pre-flight, audit pretrained empat kelas, dan training YOLO12s + L-Ghost + EMA-32.
log_section('MODEL PRE-FLIGHT — YOLO12S + L-GHOST + EMA-32')
source = YOLO('yolo12s.pt')
inspection_model = DetectionModel(str(MODEL_YAML), nc=len(CLASS_NAMES), verbose=False)
ema_layers = [layer.i for layer in inspection_model.model if isinstance(layer, EMA)]
lghost_layers = [layer.i for layer in inspection_model.model if isinstance(layer, LGhostBottleneck)]
assert ema_layers == [15,19]
assert lghost_layers == [16, 20]
assert inspection_model.model[-1].nc == 4
assert inspection_model.model[-1].f == [15, 19, 22]
preflight_transfer = inspection_model.load(source.model, verbose=False)
assert preflight_transfer['mode'] == 'semantic'
assert preflight_transfer['exact_transferred_parameter_elements'] > 8_000_000
EXPECTED_TRANSFER = {
    key: preflight_transfer[key] for key in (
        'mode', 'source_tensors', 'target_tensors', 'exact_transferred_tensors',
        'new_module_tensors', 'shape_mismatch_tensors', 'exact_transferred_parameter_elements',
    )
}
print(f'EMA layers   : {ema_layers}')
print(f'L-Ghost layers: {lghost_layers}')
inspection_model.info(verbose=True, imgsz=IMGSZ)
print(json.dumps(EXPECTED_TRANSFER, indent=2))
del inspection_model, source
if torch.cuda.is_available():
    torch.cuda.empty_cache()

TRAINING_TRANSFER_REPORT = {}

def verify_semantic_pretrained_transfer(trainer):
    report = dict(getattr(trainer.model, 'pretrained_transfer_report', {}))
    differences = {
        key: {'expected': expected, 'actual': report.get(key)}
        for key, expected in EXPECTED_TRANSFER.items() if report.get(key) != expected
    }
    assert trainer.model.model[-1].nc == 4
    if differences:
        raise AssertionError(f'Pretrained audit failed: {json.dumps(differences, indent=2)}')
    TRAINING_TRANSFER_REPORT.update(report)
    (Path(trainer.save_dir) / 'semantic_pretrained_transfer.json').write_text(
        json.dumps(report, indent=2), encoding='utf-8'
    )
    (Path(trainer.save_dir) / 'model.txt').write_text(str(trainer.model), encoding='utf-8')
    log_section('SEMANTIC PRETRAINED VERIFIED — 4 CLASSES')
    print(json.dumps(report, indent=2))

model = YOLO(str(MODEL_YAML), task='detect')
model.add_callback('on_pretrain_routine_end', verify_semantic_pretrained_transfer)
log_section('TRAINING — YOLO12S + L-GHOST + EMA-32 PRETRAINED')
model.train(
    trainer=LGhostDetectionTrainer, compile=False, time=None,
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, nbs=NBS,
    device=DEVICE, workers=WORKERS, project=str(RUNS_DIR), name=EXPERIMENT_NAME,
    exist_ok=False, pretrained='yolo12s.pt', optimizer=OPTIMIZER, lr0=LR0,
    momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, cos_lr=False,
    patience=PATIENCE, seed=SEED, deterministic=True, plots=True, verbose=True,
)
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
if not BEST_PT.is_file():
    raise FileNotFoundError(f'best.pt tidak ditemukan: {BEST_PT}')
print(f'Run directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')
assert TRAINING_TRANSFER_REPORT, 'Audit pretrained trainer tidak dijalankan.'

In [ ]:
# 4. Validation dan test menggunakan best.pt.
def metric_summary(metrics):
    return {
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'map50': float(metrics.box.map50),
        'map50_95': float(metrics.box.map),
        'map50_95_per_class': {CLASS_NAMES[i]: float(v) for i, v in enumerate(metrics.box.maps)},
        'save_dir': str(metrics.save_dir),
    }

best_model = YOLO(str(BEST_PT))
PRUNING_STAGES = [int(m.stage) for m in best_model.model.modules() if isinstance(m, LearnedGroupConv)]
print(f'LG pruning stages best.pt: {PRUNING_STAGES} (final stage = 3)')
reports = {}
for split in ('val', 'test'):
    log_section(f'{split.upper()} — BEST.PT')
    metrics = best_model.val(
        data=str(DATA_YAML), split=split, imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_{split}', exist_ok=False, plots=True,
    )
    reports[split] = metric_summary(metrics)
    print(json.dumps(reports[split], indent=2))
VAL_REPORT, TEST_REPORT = reports['val'], reports['test']

In [ ]:
# 5. Ekspor konfigurasi, metrik, checkpoint, dan plot ke ZIP.
EVALUATION_REPORT = {
    'validation': VAL_REPORT, 'test': TEST_REPORT,
    'pretrained_transfer': TRAINING_TRANSFER_REPORT, 'best_lg_pruning_stages': PRUNING_STAGES,
}
EVALUATION_JSON = WORKDIR / f'{EXPERIMENT_NAME}_evaluation_metrics.json'
EVALUATION_JSON.write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding='utf-8')
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(json.dumps({
    'model': 'YOLO12s + L-Ghost + EMA-32', 'model_yaml': str(MODEL_YAML), 'pretrained': 'yolo12s.pt',
    'repository': REPO_URL, 'source_branch': SOURCE_BRANCH, 'source_commit': REPO_COMMIT,
    'source_notebook': 'aceh-ardd-yolo12-baseline (1).ipynb',
    'source_notebook_sha256': '041497ac45989ee40e7844b0a8823c19174026dd864727315d1324fdeb7a88c5',
    'lg_groups': 4, 'lg_pruning_ratio': 0.6, 'dataset_root': str(DATA_ROOT), 'classes': CLASS_NAMES,
    'dataset_report': dataset_report, 'epochs': EPOCHS, 'imgsz': IMGSZ,
    'batch': BATCH, 'nbs': NBS, 'optimizer': OPTIMIZER, 'lr0': LR0,
    'momentum': MOMENTUM, 'weight_decay': WEIGHT_DECAY, 'seed': SEED,
    'best_checkpoint': str(BEST_PT), 'last_checkpoint': str(LAST_PT),
}, indent=2), encoding='utf-8')

ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as archive:
    archive_items = [
        RUN_DIR, Path(VAL_REPORT['save_dir']), Path(TEST_REPORT['save_dir']),
        DATA_YAML, RUN_CONFIG, EVALUATION_JSON, MODEL_YAML,
    ]
    files = []
    for item in archive_items:
        files.extend([item] if item.is_file() else [p for p in item.rglob('*') if p.is_file()])
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))

log_section('EXPERIMENT COMPLETED')
print(json.dumps(EVALUATION_REPORT, indent=2))
print(f'ZIP created: {ZIP_PATH} ({len(files)} files)')
from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))